# Imitation Learning - ACT

Imitation learning is a machine learning approach where a model is trained to mimic expert behavior by observing and replicating demonstrations, enabling it to perform tasks similarly to the expert. ACT is an action chunking policy with Transformers, an architecture designed for sequence modeling, and train it as a conditional VAE (CVAE) to capture the variability in human data. It significantly outperforms previous imitation learning algorithms on a range of simulated and real-world fine manipulation tasks. See the original paper for details: [Action Chunking Transformer](https://arxiv.org/pdf/2304.13705).

We have built an imitation learning pipeline for ACT, which can be used to train and evaluate the ACT model on different tasks. In this tutorial, we will introduce how to setup ACT pipeline from installing all the required packages to convert the model to OpenVINO and running the model.

**Note:** This notebook currently supports **Ubuntu 22.04** only.

Below is a camera viewer showcasing four different camera perspectives for the peg insertion task. From left to right: angle camera, left wrist camera, right wrist camera, and top camera.

![Camera Views](https://github.com/open-edge-platform/edge-ai-suites/raw/main/robotics-ai-suite/pipelines/act-sample/README.assets/act-sim-cameras.png)


#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Dependency and Core Installation Verification](#Dependency-and-Core-Installation-Verification)
- [Convert PyTorch model checkpoint to OpenVino IR](#Convert-PyTorch-model-checkpoint-to-OpenVino-IR)
- [Select Target Device](#Select-Target-Device)
- [Evaluate the Policy](#Evaluate-the-Policy)

### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/aloha-act/aloha-act.ipynb" />

## Prerequisites

Download our pre-trained weights from this link: [Download Link](https://eci.intel.com/embodied-sdk-docs/_downloads/sim_insertion_scripted.zip), unzip it, and place it in the `aloha-act` directory, or run the next cell to do this automatically.

In [ ]:
# Run this cell to download and extract the sim_insertion_scripted package if you haven't done so already
import requests
import zipfile
import pathlib

# Define target directory and download URL
TARGET_DIR = pathlib.Path.cwd()  # Current directory
ZIP_URL = "https://eci.intel.com/embodied-sdk-docs/_downloads/sim_insertion_scripted.zip"
ZIP_PATH = TARGET_DIR / "sim_insertion_scripted.zip"

# Create target directory if it doesn't exist
TARGET_DIR.mkdir(parents=True, exist_ok=True)

# Download the zip file
print(f"[STEP] Downloading {ZIP_URL}...")
response = requests.get(ZIP_URL, stream=True)
response.raise_for_status()

with open(ZIP_PATH, "wb") as f:
    for chunk in response.iter_content(chunk_size=8192):
        f.write(chunk)
print(f"[INFO] Downloaded to {ZIP_PATH}")

# Unzip the file
print(f"[STEP] Extracting {ZIP_PATH}...")
with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(TARGET_DIR)
print(f"[INFO] Extracted to {TARGET_DIR}")

# Remove the zip file
ZIP_PATH.unlink()
print(f"[INFO] Removed {ZIP_PATH}")
print("[DONE] Download and extraction complete.")

## Dependency and Environment Setup

Run the next cell to set up the ACT pipeline environment and install all required packages.

In [ ]:
# Environment Setup (Aloha ACT Pipeline)
import requests
import os
import sys
import subprocess
import pathlib

# Fetch notebook utilities if missing
if not pathlib.Path("notebook_utils.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py")
    open("notebook_utils.py", "w").write(r.text)

if not pathlib.Path("pip_helper.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/pip_helper.py")
    open("pip_helper.py", "w").write(r.text)

from pip_helper import pip_install

pip_install("-q", "ipywidgets")

# Sparse checkout for act-sample subdir from edge-ai-suites
TOP_REPO_URL = "https://github.com/open-edge-platform/edge-ai-suites.git"
TOP_DIR = pathlib.Path("edge-ai-suites")
SPARSE_PATH = "robotics-ai-suite/pipelines/act-sample"
REPO_DIR = TOP_DIR / SPARSE_PATH
ACT_DIR = REPO_DIR / "act"
TONYZ_ACT_URL = "https://github.com/tonyzhaozh/act.git"
TONYZ_ACT_COMMIT = "742c753c0d4a5d87076c8f69e5628c79a8cc5488"

if not TOP_DIR.exists():
    print("[STEP] Cloning repo with sparse checkout:", TOP_REPO_URL)
    env = os.environ.copy()
    env["GIT_LFS_SKIP_SMUDGE"] = "1"
    subprocess.check_call(["git", "clone", "--filter=blob:none", "--sparse", TOP_REPO_URL, str(TOP_DIR)], env=env)

# Enable cone mode and set only the act-sample path
subprocess.check_call(["git", "-C", str(TOP_DIR), "sparse-checkout", "init", "--cone"])
subprocess.check_call(["git", "-C", str(TOP_DIR), "sparse-checkout", "set", SPARSE_PATH])
print("[INFO] Checked out subdir:", REPO_DIR)

# Initialize only the 'act' submodule inside act-sample (if present)
try:
    submodule_path = str(pathlib.Path(SPARSE_PATH) / "act")
    subprocess.check_call(["git", "-C", str(TOP_DIR), "submodule", "update", "--init", "--depth", "1", submodule_path])
    print("[INFO] act submodule initialized.")
except subprocess.CalledProcessError:
    print("[WARN] act submodule init failed; proceeding with fallback clone.")

# Fallback: clone ACT repo to populate act/ if still empty
if not ACT_DIR.exists() or not any(ACT_DIR.iterdir()):
    ACT_DIR.mkdir(parents=True, exist_ok=True)
    print("[STEP] Fallback clone TonyZhao ACT into:", ACT_DIR)
    subprocess.check_call(["git", "clone", TONYZ_ACT_URL, str(ACT_DIR)])
    subprocess.check_call(["git", "-C", str(ACT_DIR), "checkout", TONYZ_ACT_COMMIT])
    print("[INFO] ACT repo checked out at commit:", TONYZ_ACT_COMMIT)

# Create __init__.py to make detr a proper Python package
DETR_DIR = ACT_DIR / "detr"
detr_init = DETR_DIR / "__init__.py"
if DETR_DIR.exists() and not detr_init.exists():
    print("[STEP] Creating __init__.py in DETR directory")
    detr_init.write_text("# DETR package\n")
    print("[INFO] __init__.py created")

# Also check if detr/models exists and has __init__.py
models_dir = DETR_DIR / "models"
models_init = models_dir / "__init__.py"
if models_dir.exists() and not models_init.exists():
    print("[STEP] Creating __init__.py in models directory")
    models_init.write_text("# DETR models\n")
    print("[INFO] models/__init__.py created")

# Apply patches from act-sample/patches
PATCHES_SUBDIRS = ["ov", "ipex"]
SKIP_PATCHES = ["0006-add-ros2-node-and-use-fixed-cube-pose.patch"]

for subdir in PATCHES_SUBDIRS:
    PATCHES_DIR = REPO_DIR / "patches" / subdir
    if PATCHES_DIR.exists():
        print(f"\n[INFO] Found patches directory: {PATCHES_DIR}")
        patch_files = sorted(PATCHES_DIR.glob("*.patch"))
        print(f"[INFO] Found {len(patch_files)} patch files in {subdir}/")

        for patch_file in patch_files:
            if patch_file.name in SKIP_PATCHES:
                print(f"\n[STEP] Skipping patch: {patch_file.name}")
                continue

            print(f"\n[STEP] Applying patch: {patch_file.name}")
            res = subprocess.run(["git", "-C", str(ACT_DIR), "apply", str(patch_file.resolve())], capture_output=True, text=True)
            if res.returncode == 0:
                print(f"[INFO] Patch applied successfully: {patch_file.name}")
            else:
                print(f"[WARN] Failed to apply patch {patch_file.name}")
                print(f"[ERROR] {res.stderr}")
    else:
        print(f"[WARN] Patches directory not found: {PATCHES_DIR}")

# Install required packages
pip_install("-U", "pip", "setuptools", "wheel")
print("[STEP] Install required packages via pip_install")

# Install torch and torchvision with CPU-only index
pip_install("-q", "--index-url", "https://download.pytorch.org/whl/cpu", "torch==2.7.1", "torchvision==0.22.1")

# Install the rest of the packages
pip_install(
    "-q",
    *[
        "pyquaternion==0.9.9",
        "pyyaml==6.0.1",
        "rospkg==1.5.0",
        "pexpect==4.8.0",
        "mujoco==3.2.6",
        "dm_control==1.0.26",
        "matplotlib==3.10.0",
        "einops==0.6.0",
        "packaging==23.0",
        "h5py==3.12.1",
        "ipython==8.12.0",
        "opencv-python==4.10.0.84",
        "transformers==4.37.0",
        "accelerate==0.23.0",
        "huggingface-hub==0.24.7",
        "openvino==2025.4.1",
        "numpy<2.0",
    ],
)

# Install DETR from the act repo (global install)
DETR_DIR = ACT_DIR / "detr"
print("[STEP] Installing DETR from:", DETR_DIR)
if DETR_DIR.exists():
    # Debug: list DETR directory structure
    print("[DEBUG] DETR_DIR contents:", list(DETR_DIR.iterdir())[:15])
    detr_src = DETR_DIR / "src"
    if detr_src.exists():
        print("[DEBUG] src/ contents:", list(detr_src.iterdir())[:15])

    subprocess.check_call([sys.executable, "-m", "pip", "install", "."], cwd=str(DETR_DIR))
    print("[INFO] DETR installed globally.")

    # Verify import
    import importlib

    try:
        detr_mod = importlib.import_module("detr")
        print("[INFO] detr import OK from:", getattr(detr_mod, "__file__", "<namespace>"))
    except Exception as e:
        print("[WARN] detr import failing in notebook kernel (may work in subprocess):", e)
else:
    print("[ERROR] DETR directory not found:", DETR_DIR)

print("[DONE] Setup complete.")

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("aloha-act.ipynb")

## Convert PyTorch Model Checkpoint to OpenVINO IR

To convert the model, run the next cell to execute `ov_convert.py`, a script that converts the PyTorch model to OpenVINO IR format. The cell runs the script with the following command:

`python3 ov_convert.py --ckpt_path <ckpt_path> --height 480 --weight 640 --camera_num 4 --chunk_size 100`


In [ ]:
import os
import sys
import subprocess
import pathlib

CKPT_PATH = pathlib.Path("sim_insertion_scripted/four_camera/policy_best.ckpt").resolve()
ACT_DIR = pathlib.Path("edge-ai-suites/robotics-ai-suite/pipelines/act-sample/act").resolve()
SCRIPT = ACT_DIR / "ov_convert.py"


if not SCRIPT.exists():
    raise FileNotFoundError(f"ov_convert.py not found: {SCRIPT}")
if not CKPT_PATH.exists():
    raise FileNotFoundError(f"Checkpoint not found: {CKPT_PATH}")

cmd = [
    sys.executable,
    str(SCRIPT),
    "--ckpt_path",
    str(CKPT_PATH),
    "--height",
    "480",
    "--weight",
    "640",
    "--camera_num",
    "4",
    "--chunk_size",
    "100",
]
print("[STEP] Running from:", ACT_DIR)
print("[STEP] Command:", " ".join(cmd))

# Create environment with ACT_DIR in PYTHONPATH
env = os.environ.copy()
env["PYTHONPATH"] = str(ACT_DIR) + os.pathsep + env.get("PYTHONPATH", "")
print("[INFO] PYTHONPATH:", env["PYTHONPATH"])

res = subprocess.run(cmd, capture_output=True, text=True, cwd=str(ACT_DIR), env=env)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise subprocess.CalledProcessError(res.returncode, cmd)

## Select Target Device

Run the next cell to select the target device for model inference.

In [ ]:
from notebook_utils import device_widget

TARGET_DEVICE = device_widget(default="CPU")
TARGET_DEVICE

## Evaluate the Policy

Run the next cell to evaluate the policy. The cell executes `imitate_episodes.py`, which runs the evaluation script with the following command:

`python3 imitate_episodes.py --task_name sim_insertion_scripted --ckpt_dir <ckpt_dir> --policy_class ACT --kl_weight 10 --chunk_size 100 --hidden_dim 512 --batch_size 8 --dim_feedforward 3200 --num_epochs 2000 --lr 1e-5 --seed 0 --device CPU/GPU --eval`

**Note:** This may take 10 to 20 minutes depending on your hardware.

Below is a demonstration of the peg insertion task:

![Peg Insertion Demo](https://github.com/open-edge-platform/edge-ai-suites/raw/main/robotics-ai-suite/pipelines/act-sample/README.assets/act-sim-insertion-demo.gif)

### Understanding the Output

The evaluation runs 10 rollout episodes and reports the following metrics:

**Per-Episode Metrics:**
- **`episode_return`**: Total cumulative reward for the entire episode (sum of all step rewards)
- **`episode_highest_reward`**: Maximum reward achieved in a single step during the episode
- **`env_max_reward`**: Maximum possible reward per step (4 for this task)
- **`Success`**: Whether the episode met the success criteria
- **`Avg fps`**: Inference speed (frames per second)

**Overall Results:**
- **Success rate**: Percentage of episodes that completed successfully
- **Average return**: Mean cumulative reward across all episodes
- **Reward breakdown**: Hierarchical achievement levels (0-4), where:
  - Reward >= 0: Basic interaction
  - Reward >= 1-3: Partial task completion
  - Reward >= 4: Full task completion (successful peg insertion)

A perfect run achieves 100% success rate with all episodes reaching reward level 4.

In [4]:
import os
import sys
import subprocess
import pathlib

CKPT_DIR = pathlib.Path("sim_insertion_scripted/four_camera").resolve()
ACT_DIR = pathlib.Path("edge-ai-suites/robotics-ai-suite/pipelines/act-sample/act").resolve()
SCRIPT = ACT_DIR / "imitate_episodes.py"

if not SCRIPT.exists():
    raise FileNotFoundError(f"imitate_episodes.py not found: {SCRIPT}")
if not CKPT_DIR.exists():
    raise FileNotFoundError(f"Checkpoint directory not found: {CKPT_DIR}")

cmd = [
    sys.executable,
    str(SCRIPT),
    "--task_name",
    "sim_insertion_scripted",
    "--ckpt_dir",
    str(CKPT_DIR),
    "--policy_class",
    "ACT",
    "--kl_weight",
    "10",
    "--chunk_size",
    "100",
    "--hidden_dim",
    "512",
    "--batch_size",
    "8",
    "--dim_feedforward",
    "3200",
    "--num_epochs",
    "2000",
    "--lr",
    "1e-5",
    "--seed",
    "0",
    "--device",
    TARGET_DEVICE.value,
    "--eval",
]

print("[STEP] Running imitate_episodes.py ...")

# Create environment with ACT_DIR in PYTHONPATH
env = os.environ.copy()
env["PYTHONPATH"] = str(ACT_DIR) + os.pathsep + env.get("PYTHONPATH", "")

# Skip evaluation on headless systems
if "DISPLAY" not in env:
    print("[INFO] No display detected (headless). Skipping evaluation cell.")
else:
    res = subprocess.run(cmd, capture_output=True, text=True, cwd=str(ACT_DIR), env=env)
    print(res.stdout)
    if res.stderr:
        print("[ERROR] STDERR:", res.stderr)
    if res.returncode != 0:
        print(f"[ERROR] Process returned exit code {res.returncode}")
        raise subprocess.CalledProcessError(res.returncode, cmd)
    print("[DONE] complete.")

[STEP] Running imitate_episodes.py ...
Avg fps 11.02142893072146
Rollout 0
episode_return=415, episode_highest_reward=4, env_max_reward=4, Success: True
Avg fps 11.07208736906833
Rollout 1
episode_return=515, episode_highest_reward=4, env_max_reward=4, Success: True
Avg fps 10.795481417256465
Rollout 2
episode_return=460, episode_highest_reward=4, env_max_reward=4, Success: True
Avg fps 11.06123185593497
Rollout 3
episode_return=456, episode_highest_reward=4, env_max_reward=4, Success: True
Avg fps 11.018667380104755
Rollout 4
episode_return=476, episode_highest_reward=4, env_max_reward=4, Success: True
Avg fps 11.146653664774483
Rollout 5
episode_return=458, episode_highest_reward=4, env_max_reward=4, Success: True
Avg fps 11.267832355940767
Rollout 6
episode_return=464, episode_highest_reward=4, env_max_reward=4, Success: True
Avg fps 11.115341265800913
Rollout 7
episode_return=479, episode_highest_reward=4, env_max_reward=4, Success: True
Avg fps 11.135617591208492
Rollout 8
episode